<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

This exercise covers two aspects:
* In tasks 1-6 you will implement mechanisms that allow training deeper models. After doing each of the tasks you can look at the plots and check how your changes impact gradients of network layers.
* In task 7 you will implement a convnet using [conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) and try to reach 99% accuracy on MNIST.

For this notebook, running on CPU is sufficient: tasks 1–5. run under a minute, 6. under two minutes, and 7. up to 5 minutes.

Tasks:
1. Check that the given implementation reaches 95% test accuracy for
   the architecture `input-64-64-10` in 8 epochs.
2. Modify `Linear.reset_parameters()` to improve initialization and
   check that the network learns much faster and reaches over 97% test accuracy.
   A good basic initialization scheme is so-called *Glorot initialization*.
   For a linear layer with $n_{in}$ and  $n_{out}$ input and output features,
   it samples each weight from a normal distribution with $0$ mean and a standard deviation of $\sqrt{\frac{2}{n_{in}+n_{out}}}$.  
   Check how that changes the distribution of gradients at the first epoch.
3. Check, that with proper initialization we can train the architecture
   `input-64-64-64-64-64-64-64-10` (seven hidden layers of `64`), for e.g. 3 epochs,
   while with the original, bad initialization, it does not even get off the ground.
4. Implement Dropout as a PyTorch module (without using torch.nn.Dropout).
   Use `p=0.25` as the zeroing probability.
   Use Dropout after activations of each layer, except the last.
5. Check that with 10 hidden layers (64 units each), even with proper initialization,
   but with dropout added, the network has a hard time to start learning.
6. Implement batch normalization (without using torch.nn.BatchNorm).
    * compute batch mean and variance
    * add new variables beta and gamma
    * check that the network learns even for 10 hidden layers.
    * check how gradients change in comparison to network without batch norm.
   
   In pseudo-code:
    ```
		if training:
		    var, mean = torch.var_mean(x, dim=/batch/)
		    update running_mean, running_var
		else:
			mean, var = running_mean, running_var
		x = (x - mean) / sqrt(var + eps)
		return x * gamma + beta
    ```

7. Design and implement a simple convolutional network and achieve 99% test accuracy (in up to 10 epochs, 5 should be enough). The architecture is up to you, but even a few convolutional layers should be enough.
You may use `torch.nn.Conv2d, BatchNorm2d, MaxPool2d, ReLU, Dropout2d, Linear, Flatten, BatchNorm1d`.
Don't make it too generic, just try a few fixed layers.

In [111]:
import math
import sys

import numpy as np
import PIL.Image
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from plotly.subplots import make_subplots
from torch import Tensor
from torch.nn import init

if 'google.colab' in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()

In [112]:
# @title Visualize gradients

class GradientVisualizer:
    def __init__(self, net: nn.Module, num_epochs: int) -> None:
        self.num_epochs = num_epochs
        self.linear_layers = self.get_linear_layers(net)

        self.grad_to_weight_fig = None
        self.grads_in_layers_fig = None
        self.grads_at_epochs_fig = None
        self.init_figures()

    @staticmethod
    def get_linear_layers(net):
        linear_layers = []
        for field in net.modules():
            if isinstance(field, nn.Linear | Linear):
                linear_layers.append(field)
        assert linear_layers, "No linear layers found. Forgot to use Sequential or ModuleList?"
        return linear_layers

    def get_epochs_for_one_layer(self):
        """
        We want to show gradient distributions from up to 7 selected epochs
        for one linear layer.
        """
        if self.num_epochs < 7:
            return list(range(self.num_epochs))
        else:
            return torch.linspace(0, self.num_epochs - 1, 7).int().tolist()

    def get_three_epochs(self):
        """
        We want to show gradients distributions from all layers at each of
        three epochs: first, middle and last.
        """
        return [0, self.num_epochs // 2, self.num_epochs - 1]

    def rgb_to_rgba(self, rgb_color, epoch):
        """
        Value of epoch parameter determines how transparent color should be
        in comparison to others.
        Colors for earlier epochs should be more transparent/less visible.
        """
        return f'rgba{rgb_color[3:-1]},{0.6 * (epoch + 1) / self.num_epochs + 0.15})'

    def init_figures(self):
        # Initialize figure with gradient to weight ratio plot
        fig = go.Figure()
        fig.update_layout(
            title='Gradient stddev to weight stddev ratio', title_x=0.5,
            xaxis_title='Epoch',
            yaxis_title='Gradient to weight ratio (log scale)',
            height=400, width=1500, margin=dict(b=10, t=60)
        )
        fig.update_yaxes(type='log')
        for i in range(len(self.linear_layers)):
            fig.add_trace(go.Scatter(
                x=[], y=[],
                mode='lines+markers', name=f'Linear layer {i}'
            ))

        self.grad_to_weight_fig = go.FigureWidget(fig)
        display(self.grad_to_weight_fig)

        # Initialize figure visualizing gradient distributions in layers
        chosen_layers = [0, len(self.linear_layers) // 2, len(self.linear_layers) - 1]
        num_rows = (len(chosen_layers) - 1) // 3 + 1
        fig = make_subplots(
            rows=num_rows, cols=3,
            subplot_titles=[f'Linear layer {i}' for i in chosen_layers],
            vertical_spacing=0.2 / num_rows
        )
        fig.update_layout(
            title='Gradient distributions by epoch (for three chosen layers)', title_x=0.5,
            height=num_rows * 400, width=1500, margin=dict(b=10, t=60)
        )

        colors, _ = px.colors.convert_colors_to_same_type(2 * px.colors.qualitative.Plotly)
        for i, layer_num in enumerate(chosen_layers):
            row = i // 3 + 1
            col = i % 3 + 1
            fig.update_xaxes(
                title_text='Gradient value', range=(-0.1, 0.1), row=row, col=col
            )
            fig.update_yaxes(
                title_text='Density (log scale)', type='log', row=row, col=col
            )

            # Create empty traces and update them later with actual gradient distributions.
            # Unfortunately, we cannot add new traces dynamically because Colab has problem
            # with widgets from plotly (traces added dynamically are rendered twice).
            for epoch in self.get_epochs_for_one_layer():
                fig.add_trace(
                    go.Scatter(
                        mode='lines', name=f'Epoch {epoch + 1}',
                        line=dict(color=self.rgb_to_rgba(colors[layer_num], epoch)),
                        legendgroup=layer_num
                    ),
                    row=row, col=col
                )

        self.grads_in_layers_fig = go.FigureWidget(fig)
        display(self.grads_in_layers_fig)

        # Initialize figure comparing gradient distributions between layers at the
        # first, middle and last epoch
        selected_epochs_indices = self.get_three_epochs()
        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=[f'Epoch {epoch + 1}' for epoch in selected_epochs_indices]
        )
        fig.update_layout(
            title='Gradient distributions by layers (for three chosen epochs)', title_x=0.5,
            height=400, width=1500, margin=dict(b=10, t=60)
        )

        for col, epoch in enumerate(selected_epochs_indices, 1):
            fig.update_yaxes(title_text='Density (log scale)', type='log', row=1, col=col)
            fig.update_xaxes(
                title_text='Gradient value',
                range=(-0.05, 0.05) if epoch != 0 else (-1, 1),
                row=1, col=col
            )

            # Create empty traces and update them later with actual gradient distributions.
            for layer_num in range(len(self.linear_layers)):
                fig.append_trace(
                    go.Scatter(
                        mode='lines', name=f'Linear layer {layer_num}',
                        line=dict(color=colors[layer_num]), showlegend=(col == 1)
                    ),
                    row=1, col=col
                )

        self.grads_at_epochs_fig = go.FigureWidget(fig)
        display(self.grads_at_epochs_fig)

    def visualize_gradients(self, lr, epoch, batch_idx):
        # It is enough to use gradients calculated for the first batch.
        if batch_idx != 0:
            return

        epoch_grads = []
        epoch_grad_to_weight_ratios = []
        for layer in self.linear_layers:
            epoch_grads.append(layer.weight.grad.flatten().detach())
            epoch_grad_to_weight_ratios.append(
                (lr * layer.weight.grad.std() / layer.weight.std()).item()
            )

        # Update figure with gradient to weight ratio plot
        assert self.grad_to_weight_fig is not None
        for i, grad_to_weight_ratio in enumerate(epoch_grad_to_weight_ratios):
            x = self.grad_to_weight_fig.data[i].x
            next_x_val = x[-1] + 1 if x else 1
            self.grad_to_weight_fig.data[i].x += (next_x_val, )
            self.grad_to_weight_fig.data[i].y += (grad_to_weight_ratio, )

        # Update figure visualizing gradient distributions in layers
        assert self.grads_in_layers_fig is not None
        chosen_layers = [0, len(self.linear_layers) // 2, len(self.linear_layers) - 1]
        selected_epochs = self.get_epochs_for_one_layer()
        if epoch in selected_epochs:
            epoch_idx = selected_epochs.index(epoch)
            for layer_num, layer_grad in enumerate(epoch_grads):
                try:
                    layer_id = chosen_layers.index(layer_num)
                except ValueError:
                    continue
                trace_idx = layer_id * len(selected_epochs) + epoch_idx
                try:
                    hy, hx = torch.histogram(layer_grad, bins=50, density=True)
                except RuntimeError:
                    continue
                hy = hy / max(hy) + 0.001
                self.grads_in_layers_fig.data[trace_idx].x = hx[:-1].tolist()
                self.grads_in_layers_fig.data[trace_idx].y = hy.tolist()


        # Update figure visualizing gradient distributions at epochs
        assert self.grads_at_epochs_fig is not None
        selected_epochs = self.get_three_epochs()
        if epoch in selected_epochs:
            epoch_idx = selected_epochs.index(epoch)
            for layer_num, layer_grad in enumerate(epoch_grads):
                trace_idx = epoch_idx * len(self.linear_layers) + layer_num
                try:
                    hy, hx = torch.histogram(layer_grad, bins=50, density=True)
                except RuntimeError:
                    continue
                hy = hy / max(hy) + 0.001
                self.grads_at_epochs_fig.data[trace_idx].x = hx[:-1].tolist()
                self.grads_at_epochs_fig.data[trace_idx].y = hy.tolist()

In [113]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, glorot_init: bool) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(Tensor(out_features, in_features))
        self.bias = nn.Parameter(Tensor(out_features))
        self.glorot_init = glorot_init
        self.reset_parameters()

    def reset_parameters(self) -> None:
        if self.glorot_init:
            # TODO Task 2.
            self.weight.data.normal_(mean=0, std=torch.sqrt( torch.tensor(2.0 / (self.in_features+self.out_features)) ))
        else:
            self.weight.data.normal_(mean=0, std=0.25)
        init.zeros_(self.bias)

    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (*B, in_features)
        Output: shape (*B, out_features)
        """
        r = x.matmul(self.weight.T)
        r += self.bias
        return r


class Net(nn.Module):
    def __init__(self, hidden_layers: int = 1, glorot_init: bool = False) -> None:
        super().__init__()
        self.fc1 = Linear(784, 64, glorot_init=glorot_init)
        self.hidden = nn.ModuleList()
        for i in range(hidden_layers):
            self.hidden.append(Linear(64, 64, glorot_init=glorot_init))
        self.fc3 = Linear(64, 10, glorot_init=glorot_init)


    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (B, H=28, W=28)
        Output: shape (B, num_classes=10)
        """
        x = x.view(-1, 28 * 28)
        x = F.relu(self.fc1(x))
        for layer in self.hidden:
            x = F.relu(layer(x))
        x = self.fc3(x)
        return x


In [114]:
class MnistTrainer:
    def __init__(self, batch_size: int = 128) -> None:
        transform = transforms.Compose([transforms.ToTensor()])
        self.trainset = torchvision.datasets.MNIST(
            root='./data', download=True, train=True, transform=transform
        )
        self.trainloader = torch.utils.data.DataLoader(
            self.trainset, batch_size=batch_size, shuffle=True, num_workers=2
        )
        self.testset = torchvision.datasets.MNIST(
            root='./data', train=False, download=True, transform=transform
        )
        self.testloader = torch.utils.data.DataLoader(
            self.testset, batch_size=1000, shuffle=False, num_workers=2
        )

    def train(
        self,
        net: nn.Module,
        gradient_visualizer: GradientVisualizer | None = None,
        epochs: int = 20,
        lr: float = 0.05,
        momentum: float = 0.9
    ) -> None:
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum)
        log_freq = 100

        for epoch in range(epochs):
            net.train()
            running_loss = 0.0
            for i, data in enumerate(self.trainloader):
                inputs, labels = data
                optimizer.zero_grad()

                outputs = net(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                if gradient_visualizer is not None:
                    gradient_visualizer.visualize_gradients(lr, epoch, i)
                optimizer.step()

                running_loss += loss.item()
                if i % log_freq == log_freq - 1:
                    running_loss /= log_freq
                    print(f"epoch {epoch + 1:>5}, batch {i + 1:>5}, loss {running_loss:.3f}")
                    running_loss = 0.0
            self.test(net)

    def test(self, net: nn.Module) -> list[float]:
        net.eval()
        correct = 0
        total = 0
        all_gt_logits = []
        with torch.no_grad():
            for data in self.testloader:
                images, labels = data
                outputs = net(images)
                gt_logits = torch.gather(outputs, dim=1, index=labels.unsqueeze(-1))
                all_gt_logits.extend(float(logit.item()) for logit in gt_logits)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        print(f"Accuracy of the network on the {total} test images: {correct / total:.2%}")
        return all_gt_logits

    def show_worst_misclassified(self, net: nn.Module, k: int = 10) -> None:
        all_gt_logits = self.test(net)
        _, indices = torch.topk(-torch.tensor(all_gt_logits), k)
        imgs = []
        labels = []
        for i in indices:
            img, label = self.testset[int(i.item())]
            imgs.append(img)
            labels.append(label)
        image = PIL.Image.fromarray(
            torch.cat(imgs, dim=-1).mul(255).to(torch.uint8).permute(1, 2, 0).numpy().squeeze()
        )
        display(image.resize((5 * image.width, 5 * image.height), PIL.Image.Resampling.NEAREST))
        print(labels)

In [115]:
epochs = 3

net = Net()
gradient_visualizer = GradientVisualizer(net, epochs)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': '0b90ea43-640c-4c62-b76c-d16aa82cff29',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': 'f98a6b58-a041-4d1d-9282-9dc0df074a31',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': 'fe4cafcf-571b-4aa0-a673-f78cdd05d6cf',
              'x': [],
              'y': []}],
    'layout': {'height': 400,
               'margin': {'b': 10, 't': 60},
               'template': '...',
               'title': {'text': 'Gradient stddev to weight stddev ratio', 'x': 0.5},
               'width': 1500,
               'xaxis': {'title': {'text': 'Epoch'}},
               'yaxis': {'title': {'text': 'G

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '61de8e52-a86e-43d0-a4c5-0456886354d7',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': 'f7682932-b348-4a3a-b3ac-075fa5cf096d',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': 'a857adf3-cbe4-41b9-880b-567d09c6fdba',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '1',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '30c402b5-da37-47b6-ad27-20c2c3a4f4c9',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'd4471b82-46ec-43a4-866c-b7e7e777a400',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'fade1fe8-4e3a-4fab-bbe5-0127221bd224',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(99, 110, 250)'},
   

KeyError: 'uid'

In [116]:
trainer = MnistTrainer(batch_size=128)
trainer.train(net, gradient_visualizer, epochs=epochs)

epoch     1, batch   100, loss 0.897
epoch     1, batch   200, loss 0.339
epoch     1, batch   300, loss 0.281
epoch     1, batch   400, loss 0.247
Accuracy of the network on the 10000 test images: 93.42%
epoch     2, batch   100, loss 0.194
epoch     2, batch   200, loss 0.182
epoch     2, batch   300, loss 0.196
epoch     2, batch   400, loss 0.163
Accuracy of the network on the 10000 test images: 94.83%
epoch     3, batch   100, loss 0.138
epoch     3, batch   200, loss 0.148
epoch     3, batch   300, loss 0.132
epoch     3, batch   400, loss 0.136
Accuracy of the network on the 10000 test images: 95.80%


### The same with Glorot initialization

In [117]:
epochs = 3

net_glorot = Net(glorot_init=True)
gradient_visualizer_glorot = GradientVisualizer(net_glorot, epochs)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': '68a6ac4e-96b3-4113-90f0-4281f5da6463',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': '9bc6fd3e-023a-4e1f-9b7a-74e140706822',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': 'ecb6f32c-3b28-4fc2-b6b9-c868f4656264',
              'x': [],
              'y': []}],
    'layout': {'height': 400,
               'margin': {'b': 10, 't': 60},
               'template': '...',
               'title': {'text': 'Gradient stddev to weight stddev ratio', 'x': 0.5},
               'width': 1500,
               'xaxis': {'title': {'text': 'Epoch'}},
               'yaxis': {'title': {'text': 'G

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': 'efa0d604-b93d-4ff1-92e3-e4f25fc6c0fe',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': '78a642de-922c-4cab-a4de-c917e205a772',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': '4b13803e-c95c-4902-8ab6-8a3141151957',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '1',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '275aeac3-478d-4af5-aeb2-2b7f649c5a3b',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2da7fe71-bf7a-4a4b-a170-5b44ddbc03e3',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b67b9434-c301-4f4c-91ce-f9b722b09592',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(99, 110, 250)'},
   

KeyError: 'uid'

KeyError: 'uid'

In [118]:
trainer_glorot = MnistTrainer(batch_size=128)
trainer_glorot.train(net_glorot, gradient_visualizer_glorot, epochs=epochs)

epoch     1, batch   100, loss 0.704
epoch     1, batch   200, loss 0.287
epoch     1, batch   300, loss 0.230
epoch     1, batch   400, loss 0.196
Accuracy of the network on the 10000 test images: 94.94%
epoch     2, batch   100, loss 0.153
epoch     2, batch   200, loss 0.135
epoch     2, batch   300, loss 0.123
epoch     2, batch   400, loss 0.128
Accuracy of the network on the 10000 test images: 96.35%
epoch     3, batch   100, loss 0.103
epoch     3, batch   200, loss 0.111
epoch     3, batch   300, loss 0.094
epoch     3, batch   400, loss 0.107
Accuracy of the network on the 10000 test images: 96.80%


### 7 hidden layers without Glorot initialization

In [ ]:
epochs = 3

net_7 = Net(hidden_layers=7, glorot_init=False)
gradient_visualizer_7 = GradientVisualizer(net_7, epochs)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': '4f932324-0cac-4f84-9fb8-d611ced739d5',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': '1c62d4ab-4f92-4250-9e67-d702fde6bce2',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': 'fa2bff0a-ebee-4987-9209-2682a81dd1fe',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': '75859f02-9bcc-4fec-aa21-32ed76889f81',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '5d8d07e4-e6c5-4584-b571-501af9ced177',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': 'd4ac0bff-2ecc-4a4a-bac3-710dc73383c4',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': '24889d49-0bb8-410a-ac7d-5814d2a13f0d',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '4',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '01d130e5-e235-4ea9-89fb-0829a4587e67',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'e5da5331-5dfc-493d-8819-8229e16ba059',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': '828a500f-af49-4a13-93c6-4d867e7b7f9e',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

KeyError: 'uid'

In [120]:
trainer_7 = MnistTrainer(batch_size=128)
trainer_7.train(net_7, gradient_visualizer_7, epochs=epochs)

epoch     1, batch   100, loss nan
epoch     1, batch   200, loss nan
epoch     1, batch   300, loss nan
epoch     1, batch   400, loss nan
Accuracy of the network on the 10000 test images: 9.80%


c:\Users\łgrzegrzółka\Documents\Bio\DeepNN\.venv\lib\site-packages\jupyter_client\session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


epoch     2, batch   100, loss nan
epoch     2, batch   200, loss nan
epoch     2, batch   300, loss nan
epoch     2, batch   400, loss nan
Accuracy of the network on the 10000 test images: 9.80%
epoch     3, batch   100, loss nan
epoch     3, batch   200, loss nan
epoch     3, batch   300, loss nan
epoch     3, batch   400, loss nan
Accuracy of the network on the 10000 test images: 9.80%


### 7 hidden layers with Glorot initialization

In [121]:
epochs = 3

net_glorot_7 = Net(hidden_layers=7, glorot_init=True)
gradient_visualizer_glorot_7 = GradientVisualizer(net_glorot_7, epochs)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': '1f85debc-e488-4fb5-abbd-f49472385f59',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': '99ca1e07-c77c-46f8-9e15-790646ecc050',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': '21bbc407-180b-48b9-9509-e80a278d9a24',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': '38570156-9388-4936-bb34-f1065ed5749d',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '64610374-7835-4ff5-96a6-e9ad3f473661',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': 'b9a73168-431b-4d80-b1b8-99cbb937fbb2',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': 'b976a1a4-f6fa-453b-b45a-f6552404c19d',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '4',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '23bebfb2-3849-453e-815d-486ece8ac5aa',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b236727e-2da4-46c7-8969-b2c4a2376724',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'fe17c522-fd4a-47b7-9bcd-3519b3dc943a',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

In [122]:
trainer_glorot_7 = MnistTrainer(batch_size=128)
trainer_glorot_7.train(net_glorot_7, gradient_visualizer_glorot_7, epochs=epochs)

epoch     1, batch   100, loss 1.671
epoch     1, batch   200, loss 0.542
epoch     1, batch   300, loss 0.313
epoch     1, batch   400, loss 0.268
Accuracy of the network on the 10000 test images: 94.50%
epoch     2, batch   100, loss 0.192
epoch     2, batch   200, loss 0.178
epoch     2, batch   300, loss 0.175
epoch     2, batch   400, loss 0.169
Accuracy of the network on the 10000 test images: 96.15%
epoch     3, batch   100, loss 0.151
epoch     3, batch   200, loss 0.130
epoch     3, batch   300, loss 0.120
epoch     3, batch   400, loss 0.133
Accuracy of the network on the 10000 test images: 95.71%


In [123]:
class Dropout(nn.Module):
    def __init__(self, p: float = 0.25) -> None:
        super().__init__()
        self.p = p

    def forward(self, x: Tensor) -> Tensor:
        """
        Zeroes `x` values with probability `self.p` and scales the rest by a `1/(1-p)` factor.
        """
        if self.training:
            keep_mask = torch.rand_like(x) >= self.p
            return x * keep_mask / (1 - self.p)
        return x

class DropNet(Net):
    def __init__(self, hidden_layers: int = 1, dropout_p: float=0.25, glorot_init: bool = False) -> None:
        super().__init__()
        self.fc1 = Linear(784, 64, glorot_init=glorot_init)
        self.hidden = nn.ModuleList()
        self.dropout = Dropout(p=dropout_p)
        for _ in range(hidden_layers):
            self.hidden.append(Linear(64, 64, glorot_init=glorot_init))
        self.fc3 = Linear(64, 10, glorot_init=glorot_init)


    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (B, H=28, W=28)
        Output: shape (B, num_classes=10)
        """
        x = x.view(-1, 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        for layer in self.hidden:
            x = F.relu(layer(x))
            x = self.dropout(x)
        x = self.fc3(x)
        return x

### Add dropout and apply it to a net with 5 and 10 hidden layers

In [124]:
epochs = 3

net_dropout_5 = DropNet(hidden_layers=5, glorot_init=True)
gradient_visualizer_dropout_5 = GradientVisualizer(net_dropout_5, epochs)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': 'f5805251-336b-4d81-bd8d-1a8bc50f9177',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': '357faab9-1968-478e-8ed2-a3fc6d5a6801',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': '5b6a4ba8-7b24-4abf-b14a-e7b780bf2521',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': '1e1b90f4-4436-4520-9975-b59b9b3cbe9f',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '596a3b63-e0c0-4170-aad2-2e359a821423',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': 'badfbaf0-79ea-4f18-8bdd-86b150f182c2',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': 'f46450e2-639a-4218-91b8-a972bb833d52',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '3',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '4b2afa9b-705b-4c03-b85b-ffe38ef7d37e',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2523a87a-c6f5-4adb-80eb-67546f4a23bc',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': '6e914890-b475-4279-9100-7df304540f12',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

In [125]:
trainer_dropout_5 = MnistTrainer(batch_size=128)
trainer_dropout_5.train(net_dropout_5, gradient_visualizer_dropout_5, epochs=epochs)

epoch     1, batch   100, loss 1.762
epoch     1, batch   200, loss 0.932
epoch     1, batch   300, loss 0.732
epoch     1, batch   400, loss 0.617
Accuracy of the network on the 10000 test images: 91.91%
epoch     2, batch   100, loss 0.506
epoch     2, batch   200, loss 0.466
epoch     2, batch   300, loss 0.479
epoch     2, batch   400, loss 0.443
Accuracy of the network on the 10000 test images: 93.34%
epoch     3, batch   100, loss 0.387
epoch     3, batch   200, loss 0.371
epoch     3, batch   300, loss 0.370
epoch     3, batch   400, loss 0.368
Accuracy of the network on the 10000 test images: 94.50%


In [126]:
epochs = 3

net_dropout_10 = DropNet(hidden_layers=10, glorot_init=True)
gradient_visualizer_dropout_10 = GradientVisualizer(net_dropout_10, epochs)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': '15b57b0a-e0ae-4d8a-ba5f-9dc2daaacb91',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': '92a09c58-288c-4509-9d03-1555eab6ba0a',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': 'a1685ee8-9ac3-4cdd-83e3-abe80c02f62b',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': '21a07c7d-d261-46d1-8cce-e40b507c3230',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '4c051b8c-2e79-4e96-bbd4-991757278b6d',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': '53a7a00e-1992-4a75-b586-38db38849f21',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': 'e7df2933-20c6-4cb8-85de-19604bc75107',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '6',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '277e9d39-06fc-4fe9-9431-114614066a55',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': '512d787c-3cc7-4615-b0cc-2a36106c1a89',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b99d139d-694a-4388-9d3a-6ceda8e108bc',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

In [127]:
trainer_dropout_10 = MnistTrainer(batch_size=128)
trainer_dropout_10.train(net_dropout_10, gradient_visualizer_dropout_10, epochs=epochs)

epoch     1, batch   100, loss 2.274
epoch     1, batch   200, loss 1.860
epoch     1, batch   300, loss 1.659
epoch     1, batch   400, loss 1.586
Accuracy of the network on the 10000 test images: 27.28%
epoch     2, batch   100, loss 1.512
epoch     2, batch   200, loss 1.522
epoch     2, batch   300, loss 1.525
epoch     2, batch   400, loss 1.465
Accuracy of the network on the 10000 test images: 39.14%
epoch     3, batch   100, loss 1.480
epoch     3, batch   200, loss 1.472
epoch     3, batch   300, loss 1.463
epoch     3, batch   400, loss 1.394
Accuracy of the network on the 10000 test images: 47.31%


### Batchnorm

In [128]:
class BatchNorm(nn.Module):
    def __init__(self, num_features: int, momentum: float=0.1, eps: float=1e-6) -> None:
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        self.momentum = momentum
        self.eps = eps
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))


    def forward(self, x: Tensor) -> Tensor:
        """
        Computes BatchNorm for a batch.
        """
        if self.training:
            var, mean = torch.var_mean(x, dim=0, correction=0)
            with torch.no_grad():
                self.running_mean = (1-self.momentum) * self.running_mean + self.momentum * mean
                self.running_var = (1-self.momentum) * self.running_var + self.momentum * var
        else:
            mean, var = self.running_mean, self.running_var
        x = (x-mean) / torch.sqrt(var + self.eps)
        x = self.gamma*x + self.beta
        return x


class BatchNet(Net):
    def __init__(self, hidden_layers: int = 1, dropout_p: float=0.25, glorot_init: bool = False) -> None:
        super().__init__()
        self.fc1 = Linear(784, 64, glorot_init=glorot_init)
        self.hidden = nn.ModuleList()
        self.batchnorms = nn.ModuleList([BatchNorm(64)])
        self.dropout = Dropout(p=dropout_p)
        for _ in range(hidden_layers):
            self.hidden.append(Linear(64, 64, glorot_init=glorot_init))
            self.batchnorms.append(BatchNorm(64))
        self.fc3 = Linear(64, 10, glorot_init=glorot_init)


    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (B, H=28, W=28)
        Output: shape (B, num_classes=10)
        """
        x = x.view(-1, 28 * 28)
        x = F.relu( self.batchnorms[0](self.fc1(x)))
        x = self.dropout(x)
        for layer, batchnorm in zip(self.hidden, self.batchnorms[1:], strict=True):
            x = F.relu( batchnorm( layer(x) ) )
            x = self.dropout(x)
        x = self.fc3(x)
        return x

In [129]:
epochs = 3

net_batchnorm_10 = BatchNet(hidden_layers=10, glorot_init=True)
gradient_visualizer_batchnorm_10 = GradientVisualizer(net_batchnorm_10, epochs)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': 'bb5c120a-503e-45ca-bb42-55234039e1d5',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': '19d76e76-e8b1-4b3c-80b4-bef2d0e994d3',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': '306455b8-3a48-4735-9bf5-5afb62ebbe2c',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': 'cb2a0909-f78a-4be5-a252-418088c34eff',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '72108afc-60e0-40e6-8ef9-1ce49513d478',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': 'd10168ab-3f5f-4596-9d00-da46bbbf036e',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': '5b6a2622-32f2-4ea5-82af-08bb51159988',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '6',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '881a5c2d-0719-4b55-b918-80e55224dfe9',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c1770b50-4f16-484a-abcb-00309a82d86e',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b11a1c96-25dc-4f66-8ada-04761e2da6d3',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

In [130]:
trainer_batchnorm_10 = MnistTrainer(batch_size=128)
trainer_batchnorm_10.train(net_batchnorm_10, gradient_visualizer_batchnorm_10, epochs=epochs)

epoch     1, batch   100, loss 2.173
epoch     1, batch   200, loss 1.600
epoch     1, batch   300, loss 1.259
epoch     1, batch   400, loss 1.146
Accuracy of the network on the 10000 test images: 57.93%
epoch     2, batch   100, loss 1.031
epoch     2, batch   200, loss 0.994
epoch     2, batch   300, loss 0.941
epoch     2, batch   400, loss 0.858
Accuracy of the network on the 10000 test images: 75.51%
epoch     3, batch   100, loss 0.819
epoch     3, batch   200, loss 0.791
epoch     3, batch   300, loss 0.774
epoch     3, batch   400, loss 0.741
Accuracy of the network on the 10000 test images: 75.92%


### Let's go with it all:)

This exercise covers two aspects:
* In tasks 1-6 you will implement mechanisms that allow training deeper models. After doing each of the tasks you can look at the plots and check how your changes impact gradients of network layers.
* In task 7 you will implement a convnet using [conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) and try to reach 99% accuracy on MNIST.

For this notebook, running on CPU is sufficient: tasks 1–5. run under a minute, 6. under two minutes, and 7. up to 5 minutes.

Tasks:
1. Check that the given implementation reaches 95% test accuracy for
   the architecture `input-64-64-10` in 8 epochs.
2. Modify `Linear.reset_parameters()` to improve initialization and
   check that the network learns much faster and reaches over 97% test accuracy.
   A good basic initialization scheme is so-called *Glorot initialization*.
   For a linear layer with $n_{in}$ and  $n_{out}$ input and output features,
   it samples each weight from a normal distribution with $0$ mean and a standard deviation of $\sqrt{\frac{2}{n_{in}+n_{out}}}$.  
   Check how that changes the distribution of gradients at the first epoch.
3. Check, that with proper initialization we can train the architecture
   `input-64-64-64-64-64-64-64-10` (seven hidden layers of `64`), for e.g. 3 epochs,
   while with the original, bad initialization, it does not even get off the ground.
4. Implement Dropout as a PyTorch module (without using torch.nn.Dropout).
   Use `p=0.25` as the zeroing probability.
   Use Dropout after activations of each layer, except the last.
5. Check that with 10 hidden layers (64 units each), even with proper initialization,
   but with dropout added, the network has a hard time to start learning.
6. Implement batch normalization (without using torch.nn.BatchNorm).
    * compute batch mean and variance
    * add new variables beta and gamma
    * check that the network learns even for 10 hidden layers.
    * check how gradients change in comparison to network without batch norm.
   
   In pseudo-code:
    ```
		if training:
		    var, mean = torch.var_mean(x, dim=/batch/)
		    update running_mean, running_var
		else:
			mean, var = running_mean, running_var
		x = (x - mean) / sqrt(var + eps)
		return x * gamma + beta
    ```

7. Design and implement a simple convolutional network and achieve 99% test accuracy (in up to 10 epochs, 5 should be enough). The architecture is up to you, but even a few convolutional layers should be enough.
You may use `torch.nn.Conv2d, BatchNorm2d, MaxPool2d, ReLU, Dropout2d, Linear, Flatten, BatchNorm1d`.
Don't make it too generic, just try a few fixed layers.

In [135]:
class ConvNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.features = nn.Sequential(
            # (B, 1, 28, 28) -> (B, 32, 14, 14)
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # (B, 32, 14, 14) -> (B, 64, 14, 14)
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # (B, 64, 14, 14) -> (B, 64, 7, 7)
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        return self.classifier(x)

In [137]:
epochs = 5

net_convnet_5 = ConvNet()
gradient_visualizer_convnet_5 = GradientVisualizer(net_convnet_5, epochs)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': '86c61b94-c91a-43d8-a11a-d5383fc988d8',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': '74344e90-5f2c-41bd-9209-882356fe480f',
              'x': [],
              'y': []}],
    'layout': {'height': 400,
               'margin': {'b': 10, 't': 60},
               'template': '...',
               'title': {'text': 'Gradient stddev to weight stddev ratio', 'x': 0.5},
               'width': 1500,
               'xaxis': {'title': {'text': 'Epoch'}},
               'yaxis': {'title': {'text': 'Gradient to weight ratio (log scale)'}, 'type': 'log'}}
})

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.27)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '1227de9e-917e-4650-9ba5-4207b1dd4818',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.39)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': 'c2712346-5eda-413d-bab3-837ba9875ba3',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.51)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': 'd8f18dcd-865e-4b72-a885-24bc54713ee4',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'd5ea6b8f-ba18-4ab1-9d91-0c085b330746',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c7c7a055-c2a7-48c6-ac99-383c4f1342df',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': False,
              'type': 'scatter',
              'uid': '83b10d88-b294-4d56-a630-d613946436e6',
              'xaxis': 'x2',
              'yaxis': 'y2'},
             {'line': {'color': 'rgb(239, 85, 59)'},


KeyError: 'uid'

In [138]:
trainer_convnet_5 = MnistTrainer(batch_size=128)
trainer_convnet_5.train(net_convnet_5, gradient_visualizer_convnet_5, epochs=epochs)

epoch     1, batch   100, loss 0.392
epoch     1, batch   200, loss 0.108
epoch     1, batch   300, loss 0.098
epoch     1, batch   400, loss 0.061
Accuracy of the network on the 10000 test images: 98.53%
epoch     2, batch   100, loss 0.050
epoch     2, batch   200, loss 0.039
epoch     2, batch   300, loss 0.045
epoch     2, batch   400, loss 0.033
Accuracy of the network on the 10000 test images: 98.96%
epoch     3, batch   100, loss 0.024
epoch     3, batch   200, loss 0.030
epoch     3, batch   300, loss 0.029
epoch     3, batch   400, loss 0.023
Accuracy of the network on the 10000 test images: 98.97%
epoch     4, batch   100, loss 0.018
epoch     4, batch   200, loss 0.030
epoch     4, batch   300, loss 0.017
epoch     4, batch   400, loss 0.022
Accuracy of the network on the 10000 test images: 99.18%
epoch     5, batch   100, loss 0.012
epoch     5, batch   200, loss 0.015
epoch     5, batch   300, loss 0.019
epoch     5, batch   400, loss 0.018
Accuracy of the network on the 1